# Generalized AMD-stability criterion — example

Companion notebook for **amdstability**.

Everything goes through a single object, `AMDClassifier`, which handles
**any number of planets**:

| N | what happens |
|---|---|
| 1 | trivially stable ($\beta = 0$) |
| 2 | one pair, no three-body term — analytic critical AMD (Hadden & Lithwick 2018; Tamayo et al. 2021) |
| 3 | the general criterion (constrained minimization) |
| >3 | one result per adjacent triplet; the verdict is that of the most constraining one |

For each subsystem we get the critical eccentricity-like vector $c$, the
**critical** AMD, and $\beta = \mathrm{AMD}/\mathrm{AMD_{crit}}$ ($\beta<1$ stable).
The total AMD of the system is used for every subsystem, since the AMD can be
exchanged between all the planets.

In [6]:
import os
import sys

import numpy as np

from amdstability import AMDClassifier

clf = AMDClassifier()          # criterion="full", method="SLSQP"
clf

AMDClassifier(criterion='full', method='SLSQP')

## A. Three planets, from periods and masses

Normalized units: central mass `m0=1`, `G=1`. Planet masses are then in units
of the central mass. Passing `eccs` lets the classifier compute the *actual*
AMD, and therefore `beta`. Without `eccs`, only `AMD_crit` is computed.

In [7]:
res = clf.AMDcritfromelements(
    periods=[1.0, 1.4, 1.8],
    masses=[1e-5, 1e-5, 1e-5],
    eccs=[0.01, 0.005, 0.01],
)

print(res)              # human-readable summary

AMD-stability [general (full)]  N = 3
  AMD      = 6.7637e-10
  AMD_crit = 4.3108e-09
  beta     = 1.5690e-01  ->  STABLE


In [8]:
res.limiting.c, res.AMD, res.AMD_crit, res.beta, res.stable

(array([0.00785098, 0.02981638, 0.02096631]),
 6.76369744078953e-10,
 4.310753554871839e-09,
 0.15690290235092363,
 True)

The solution saturates $\tau = 1$ to machine precision. When all the $c_i$
are strictly positive, it also satisfies the first-order optimality condition
$\Lambda_2 c_2 = \Lambda_1 c_1 + \Lambda_3 c_3$, which follows from
$\tau_{12}$ depending only on $c_1+c_2$ and $\tau_{23}$ only on $c_2+c_3$:

In [9]:
from amdstability.amd import lambdas_from_periods
from amdstability.opticaldepth import TripletGeometry

L = lambdas_from_periods([1.0, 1.4, 1.8], [1e-5] * 3)
g = TripletGeometry.from_triplet(1.0, [1e-5] * 3, [1.0, 1.4, 1.8])
c = res.limiting.c

print("tau(c)                =", g.tau(c))
print("L2*c2                 =", L[1] * c[1])
print("L1*c1 + L3*c3         =", L[0] * c[0] + L[2] * c[2])
print("tau_123 (3-body term) =", g.tau_3body)

tau(c)                = 0.9999999999999996
L2*c2                 = 1.807607940649316e-07
L1*c1 + L3*c3         = 1.8076079761601137e-07
tau_123 (3-body term) = 0.15194191154102715


### An unstable configuration

In [10]:
print(clf.AMDcritfromelements(
    periods=[1.0, 1.5, 2.0],
    masses=[1e-4, 1e-5, 1e-5],
    eccs=[0.03, 0.04, 0.03],
))

AMD-stability [general (full)]  N = 3
  AMD      = 3.2430e-08
  AMD_crit = 4.9423e-09
  beta     = 6.5617e+00  ->  UNSTABLE


## B. Two planets

With a single pair there is no three-body term and the critical AMD is
analytic (Tamayo et al. 2021, appendix, with the critical eccentricity of
Hadden & Lithwick 2018):

$$
\mathrm{AMD_{crit}} = \tfrac12\,\Lambda_r\,\delta^2\,
\exp\!\left[-\left(\frac{3}{S}\right)^{4/3}\right],
\qquad
\Lambda_r = \frac{\Lambda_1\Lambda_2}{\Lambda_1+\Lambda_2},
\quad \delta = 1-\frac{a_1}{a_2},
\quad S = \frac{\delta}{\epsilon^{1/4}} .
$$

The critical eccentricities follow from minimizing
$\tfrac12(\Lambda_1 c_1^2+\Lambda_2 c_2^2)$ at fixed $c_1+c_2 = e_\mathrm{crit}$:
$c_1 = \Lambda_2\, e_\mathrm{crit}/(\Lambda_1+\Lambda_2)$,
$c_2 = \Lambda_1\, e_\mathrm{crit}/(\Lambda_1+\Lambda_2)$. No optimizer is involved.

In [11]:
res2 = clf.AMDcritfromelements(
    periods=[1.0, 1.6],
    masses=[1e-5, 1e-5],
    eccs=[0.01, 0.01],
)
print(res2)
print("c =", res2.limiting.c)

AMD-stability [general]  N = 2
  AMD      = 5.8790e-10
  AMD_crit = 5.3743e-08
  beta     = 1.0939e-02  ->  STABLE
c = [0.10340326 0.08840854]


## C. More than three planets

The criterion is applied to every set of three adjacent planets. The system
verdict is that of the most constraining triplet (smallest $\mathrm{AMD_{crit}}$,
i.e. largest $\beta$), and all the individual results stay accessible in
`res.subsystems`.

In [12]:
res5 = clf.AMDcritfromelements(
    periods=[1.0, 1.4, 1.9, 2.6, 3.4],
    masses=[1e-5, 2e-5, 1e-5, 3e-5, 1e-5],
    eccs=[0.01, 0.01, 0.02, 0.01, 0.01],
)
print(res5)

AMD-stability [general (full)]  N = 5
  AMD      = 3.7450e-09
  AMD_crit = 4.2427e-09
  beta     = 8.8271e-01  ->  STABLE
  subsystems:
    (0, 1, 2): AMD_crit = 9.4409e-09  beta = 3.9668e-01
    (1, 2, 3): AMD_crit = 4.2427e-09  beta = 8.8271e-01  <- limiting
    (2, 3, 4): AMD_crit = 4.7375e-09  beta = 7.9051e-01


In [13]:
print("limiting triplet:", res5.limiting.indices)
print()
for s in res5.subsystems:
    print(f"{s.indices}  AMD_crit={s.AMD_crit:.3e}  beta={s.beta:.3f}  "
          f"{'stable' if s.stable else 'UNSTABLE'}  c={np.round(s.c, 4)}")

limiting triplet: (1, 2, 3)

(0, 1, 2)  AMD_crit=9.441e-09  beta=0.397  stable  c=[0.0245 0.0278 0.0305]
(1, 2, 3)  AMD_crit=4.243e-09  beta=0.883  stable  c=[0.0089 0.0323 0.0049]
(2, 3, 4)  AMD_crit=4.737e-09  beta=0.791  stable  c=[0.0159 0.0129 0.0223]


In [14]:
import pandas as pd

pd.DataFrame(res5.to_dict()["subsystems"])

,indices,AMD_crit,beta,c,tau_residual,method,degenerate
0,"(0, 1, 2)",9.440873e-09,0.396684,"[0.024517668632036523, 0.02782891600227025, 0....",0.000000e+00,SLSQP,False
1,"(1, 2, 3)",4.242685e-09,0.882707,"[0.008876644328138134, 0.03228836885831534, 0....",1.110223e-16,SLSQP,False
2,"(2, 3, 4)",4.737481e-09,0.790514,"[0.015853628311140355, 0.012902547441644782, 0...",0.000000e+00,SLSQP,False


## D. From a rebound simulation

`AMDcritfromsimulation` reads masses and periods from the simulation (star =
particle 0, planets ordered from inner to outer) and computes the actual AMD
with `amd_from_sim`. It returns the same result object.
By default the AMD is planar; use `with_inclination=True` to include $\cos i$.

> Requires `rebound`: `pip install rebound`

In [15]:
try:
    import rebound as rb

    sim = rb.Simulation()
    sim.G = 1.0
    sim.add(m=1.0)                       # star
    sim.add(m=1e-5, P=1.0, e=0.01)       # planet 1
    sim.add(m=1e-5, P=1.3, e=0.005)      # planet 2
    sim.add(m=1e-5, P=1.7, e=0.01)       # planet 3
    sim.move_to_com()

    res_sim = clf.AMDcritfromsimulation(sim, with_inclination=False)
    print(res_sim)
    print()
    print("stable ->", res_sim.stable)

except ImportError:
    print("rebound not installed - skipping section D.")

AMD-stability [general (full)]  N = 3
  AMD      = 6.6830e-10
  AMD_crit = 2.6550e-09
  beta     = 2.5171e-01  ->  STABLE

stable -> True


The other criteria (section E) take orbital elements. `elements_from_simulation`
extracts them from a simulation:

In [16]:
try:
    from amdstability import elements_from_simulation

    print(clf.HillAMDcrit(**elements_from_simulation(sim)))
except NameError:
    print("rebound not installed - skipping.")

AMD-stability [Hill]  N = 3
  AMD      = 6.6830e-10
  AMD_crit = 2.7927e-08
  beta     = 2.3931e-02  ->  STABLE
  subsystems:
    (0, 1): AMD_crit = 2.7927e-08  beta = 2.3931e-02  <- limiting
    (1, 2): AMD_crit = 3.2118e-08  beta = 2.0808e-02


## E. The other AMD criteria

The same classifier exposes the classical two-planet criteria, applied to every
adjacent pair, with the same result object:

- `HillAMDcrit` — Hill stability (Petit, Laskar & Boué 2018),
- `CollisionAMDcrit` — collision (Laskar & Petit 2017),
- `MMRAMDcrit` — MMR overlap (Petit, Laskar & Boué 2017).

In [17]:
kw = dict(periods=[1.0, 1.4, 1.8], masses=[1e-5] * 3, eccs=[0.01, 0.005, 0.01])

print(f"{'AMDcritfromelements':20s} beta = {clf.AMDcritfromelements(**kw).beta:.4g}")
for name in ("HillAMDcrit", "CollisionAMDcrit", "MMRAMDcrit"):
    r = getattr(clf, name)(**kw)
    print(f"{name:20s} beta = {r.beta:.4g}  ({'stable' if r.stable else 'UNSTABLE'})")

AMDcritfromelements  beta = 0.1569
HillAMDcrit          beta = 0.02398  (stable)
CollisionAMDcrit     beta = 0.0153  (stable)
MMRAMDcrit           beta = 0.0005803  (stable)


## F. Edge cases

**Degenerate configuration.** $\tau(c=0)=\tau_{123}$: when the three-body term
alone already reaches 1, the criterion is saturated by circular orbits and
$\mathrm{AMD_{crit}}=0$. This is detected *before* the optimization — no
exception, just `degenerate=True` and $\beta=\infty$.

In [18]:
res_deg = clf.AMDcritfromelements(
    periods=[1.0, 1.05, 1.1],
    masses=[1e-4, 1e-4, 1e-4],
    eccs=[0.01, 0.01, 0.01],
)
print(res_deg)
print()
g_deg = TripletGeometry.from_triplet(1.0, [1e-4] * 3, [1.0, 1.05, 1.1])
print("degenerate:", res_deg.limiting.degenerate, "| tau_123 =", g_deg.tau_3body)

AMD-stability [general (full)]  N = 3
  AMD      = 8.2610e-09
  AMD_crit = 0.0000e+00
  beta     = inf  ->  UNSTABLE

degenerate: True | tau_123 = 1675.86899163703


**Single planet.** Trivially stable:

In [19]:
res1 = clf.AMDcritfromelements(periods=[1.0], masses=[1e-5], eccs=[0.1])
res1.beta, res1.stable

(0.0, True)

**Ordering.** Planets must be given from inner to outer; otherwise a clear
error is raised.

In [20]:
try:
    clf.AMDcritfromelements(periods=[1.0, 0.8, 1.9], masses=[1e-5] * 3)
except ValueError as e:
    print("Caught expected error:")
    print(" ", e)

Caught expected error:
  Planets must be ordered from inner to outer (strictly increasing periods).


## G. Solver options

For three planets, the minimization is solved by `SLSQP` (default) or
`trust-constr`. These solvers can stop on a point of the constraint surface
that is *not* the minimum while reporting success, which **overestimates**
$\mathrm{AMD_{crit}}$. The optimization is therefore started from many
initial conditions — the fixed `_IC_SETS` plus a grid spread over the
surface $\tau = 1$ — and the lowest feasible AMD is kept.
`n_ic_agree` counts the initial conditions that reached that minimum.

In [21]:
import warnings
from amdstability import critical_amd_triplet, AMDOptimizationError

warnings.filterwarnings("ignore", module="scipy.optimize")   # trust-constr quasi-Newton notices

for method in ("SLSQP", "trust-constr"):
    try:
        o = critical_amd_triplet([1.0, 1.4, 1.8], [1e-5] * 3, method=method)
        print(f"{method:14s} AMD_crit = {o['AMD_crit']:.6e}   "
              f"|tau-1| = {o['tau_residual']:.2e}   n_ic_agree = {o['n_ic_agree']:2d}   "
              f"c = {np.round(o['c'], 5)}")
    except AMDOptimizationError as e:
        print(f"{method:14s} failed: {e}")

SLSQP          AMD_crit = 4.310754e-09   |tau-1| = 4.44e-16   n_ic_agree = 35   c = [0.00785 0.02982 0.02097]
trust-constr   AMD_crit = 4.310754e-09   |tau-1| = 2.22e-16   n_ic_agree = 14   c = [0.00785 0.02982 0.02097]


Only the two-body terms can be used as well (`criterion="2body"`):

In [22]:
print(AMDClassifier(criterion="2body").AMDcritfromelements(
    periods=[1.0, 1.4, 1.8], masses=[1e-5] * 3, eccs=[0.01, 0.005, 0.01],
))

AMD-stability [general (2body)]  N = 3
  AMD      = 6.7637e-10
  AMD_crit = 5.6568e-09
  beta     = 1.1957e-01  ->  STABLE
